# KEEPCODING · AI Engineering · M1 · Agentes con LangChain

## Clase 5 · Lab 1 · Agente ReAct: "El contador de recompensas"

En este laboratorio construimos un **agente ReAct** con LangChain y LangGraph. Un agente ReAct razona y actúa en un bucle: piensa qué hacer (*Thought*), llama a una herramienta (*Action*), observa el resultado (*Observation*) y repite hasta resolver la tarea.

Para hacerlo divertido usamos el universo de **One Piece**: el agente buscará las recompensas de los Sombrero de Paja, sumará cifras gigantescas con una calculadora y consultará Wikipedia.

### Qué vas a aprender

- Definir **herramientas** (tools) con el decorador `@tool`.
- Conectar un modelo de chat con `bind_tools` para que elija herramientas.
- Montar un agente con `create_agent` de `langchain.agents` (en versiones antiguas, `create_react_agent` de `langgraph.prebuilt`, deprecado en LangChain v1.0).
- Ver el bucle **Thought -> Action -> Observation** en directo con `agent.stream(...)`.
- Dar **memoria conversacional** al agente con `MemorySaver` y `thread_id`.

### Qué necesitas

- Una clave de OpenAI: la variable de entorno `OPENAI_API_KEY`.
- Este notebook se ejecuta de arriba abajo en **Google Colab** o en **Jupyter** local.
- Conexión a internet (las tools consultan APIs externas; si fallan, hay un plan B).

> Nota: ejecuta las celdas en orden. Cada celda depende de las anteriores.

### Por qué importa

Un agente = LLM + tools + bucle. Es el patrón base de **toda** la clase 5: el ReAct prebuilt de hoy es el "hola mundo" del que parten ADK (M2), A2A (M3) y tu propio harness construido a mano (M5).

### Conecta con

- **Slides M1** · "De chatbot a agente", ReAct.
- **Lab 2** · el mismo agente, pero como grafo explícito (LangGraph).
- **M5** · cuando construyas este bucle sin framework, línea a línea.
- **M7** · observabilidad y trazas de agentes (Langfuse).

### Stack

`langchain>=1.0` · `langgraph>=1.0` · `langchain-openai` · `gpt-5-mini`.

### Hardware

Cualquier CPU; Colab Free sobra (no usa GPU, todo es API).


## Guion del laboratorio

1. **Setup**: instalar dependencias y cargar la API key.
2. **Tres herramientas** con `@tool`: buscar personaje (One Piece API), calculadora segura y Wikipedia.
3. **El modelo elige tool**: `ChatOpenAI` + `bind_tools` en una pregunta sencilla.
4. **El agente ReAct**: `create_agent` de LangChain v1 (sustituye a `create_react_agent`, deprecado).
5. **Demo estrella**: la recompensa total de la tripulación, viendo el bucle paso a paso.
6. **Memoria**: recompilar con `MemorySaver` y hacer una pregunta de seguimiento.
7. **Recap y retos** para practicar por tu cuenta.


## 1. Setup

### 1.1. Instalar dependencias

Instalamos las librerías del ecosistema LangChain:

- `langchain` y `langchain-openai`: el framework y el conector con OpenAI.
- `langgraph`: el runtime sobre el que corre el agente y la memoria por hilos.

Para llamar a la API de One Piece y a Wikipedia usamos `urllib`, de la librería estándar de Python: no hace falta instalar nada más.

> En Colab esto tarda unos segundos. La opción `-q` reduce el ruido en la salida.


In [1]:
# Instalación de dependencias (ejecutar una sola vez por sesión)
%pip install -q "langchain>=1.0" "langchain-openai" "langgraph>=1.0"

### 1.2. Cargar la API key de OpenAI

El agente necesita `OPENAI_API_KEY`. Esta celda intenta tres fuentes en orden:

1. La variable de entorno (si ya la tienes configurada).
2. El gestor de secretos de Google Colab (`google.colab.userdata`).
3. Un cuadro de entrada seguro con `getpass` (no muestra la clave en pantalla).

Así el notebook funciona tanto en Colab como en Jupyter local sin tocar el código.


In [2]:
import os
from getpass import getpass


def cargar_api_key() -> None:
    """Asegura que OPENAI_API_KEY esté disponible en el entorno."""
    if os.environ.get("OPENAI_API_KEY"):
        print("OPENAI_API_KEY encontrada en las variables de entorno.")
        return

    # Intento 2: secretos de Google Colab.
    try:
        from google.colab import userdata

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        print("OPENAI_API_KEY cargada desde los secretos de Colab.")
        return
    except Exception:
        pass

    # Intento 3: entrada manual segura.
    os.environ["OPENAI_API_KEY"] = getpass("Introduce tu OPENAI_API_KEY: ")
    print("OPENAI_API_KEY cargada desde la entrada manual.")


cargar_api_key()

OPENAI_API_KEY encontrada en las variables de entorno.


## 2. Las herramientas (tools)

Una **herramienta** es una función de Python que el agente puede decidir ejecutar. En LangChain se crea con el decorador `@tool`.

Dos detalles clave de una buena tool:

- El **nombre** de la función: el modelo lo usa para referirse a ella.
- El **docstring**: es la descripción que ve el modelo para decidir cuándo usarla. Escríbelo claro, como si fuera un manual de instrucciones.

Vamos a crear tres tools:

| Tool | Qué hace |
|------|----------|
| `buscar_personaje` | Devuelve la recompensa de un personaje de One Piece |
| `calculadora` | Evalúa una expresión aritmética de forma segura |
| `buscar_wikipedia` | Resume un artículo de Wikipedia |

### 2.1. Tool `buscar_personaje`

Esta tool intenta primero la **API en vivo** de One Piece (`api-onepiece.com`). Si la API falla (timeout, error, caída del servicio), cae a un **diccionario local** con las recompensas canónicas post-Wano de los 10 Sombrero de Paja.

Este patrón de *fallback* es importante en producción: una tool nunca debería romper el agente solo porque un servicio externo no responda.


In [3]:
import urllib.request
import json
import unicodedata
from langchain_core.tools import tool

# Cache en memoria: la lista completa de personajes se descarga una sola vez.
_OP_CACHE = []

# Plan B: recompensas canónicas post-Wano de los 10 Sombrero de Paja.
_OP_FALLBACK = {
    "luffy": ("Monkey D Luffy", "3.000.000.000"),
    "zoro": ("Roronoa Zoro", "1.111.000.000"),
    "nami": ("Nami", "366.000.000"),
    "sanji": ("Sanji", "1.032.000.000"),
    "usopp": ("Usopp", "500.000.000"),
    "chopper": ("Tony-Tony Chopper", "1.000"),
    "robin": ("Nico Robin", "930.000.000"),
    "franky": ("Franky", "394.000.000"),
    "brook": ("Brook", "383.000.000"),
    "jinbe": ("Jinbe", "1.100.000.000"),
}


def _normalizar(texto: str) -> str:
    """Minúsculas, sin acentos ni puntuación, espacios colapsados."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode()
    t = "".join(c if c.isalnum() else " " for c in t.lower())
    return " ".join(t.split())


def _coincide(consulta: str, nombre: str) -> bool:
    """True si 'consulta' identifica a 'nombre'. Tolera puntos, acentos y orden.

    El LLM puede pasar "Luffy" o "Monkey D. Luffy"; la API guarda
    "Monkey D Luffy" (sin punto). Normalizamos ambos y comparamos por
    contención o por palabras, nunca con un substring frágil.
    """
    q, n = _normalizar(consulta), _normalizar(nombre)
    if not q or not n:
        return False
    if q == n or q in n or n in q:
        return True
    palabras = [w for w in q.split() if len(w) >= 3]
    return bool(palabras) and all(w in n.split() for w in palabras)


def _cargar_personajes_op():
    """Descarga (una sola vez) la lista completa de personajes de One Piece."""
    global _OP_CACHE
    if _OP_CACHE:
        return _OP_CACHE
    url = "https://api.api-onepiece.com/v2/characters/en"
    req = urllib.request.Request(url, headers={"User-Agent": "keepcoding-ai-lab/1.0"})
    with urllib.request.urlopen(req, timeout=25) as r:
        _OP_CACHE = json.load(r)
    return _OP_CACHE


@tool
def buscar_personaje(nombre: str) -> str:
    """Busca la recompensa (bounty) de un personaje de One Piece por su nombre.

    Intenta primero la API en vivo de One Piece y, si falla, usa una base de
    datos local con las recompensas canónicas de la tripulación de Sombrero
    de Paja. Devuelve un texto con el nombre y la recompensa en berries.
    """
    # Intento 1: API en vivo (lista completa cacheada, se filtra en Python).
    try:
        for c in _cargar_personajes_op():
            if _coincide(nombre, c.get("name") or ""):
                return (
                    f"{c['name']}: recompensa "
                    f"{c.get('bounty') or 'desconocida'} berries (One Piece API)."
                )
        return f"No se encontró a '{nombre}' en la One Piece API."
    except Exception as e:
        # Cualquier fallo de red o de datos nos lleva al plan B.
        for k, (n, b) in _OP_FALLBACK.items():
            if _coincide(nombre, k) or _coincide(nombre, n):
                return f"{n}: recompensa {b} berries (fallback local)."
        return f"No se pudo consultar la API ni el fallback para '{nombre}': {e}"


# Probamos la tool de forma directa con .invoke()
print(buscar_personaje.invoke({"nombre": "Zoro"}))
buscar_personaje.name

Roronoa Zoro: recompensa 1.101.000.000 berries (One Piece API).


'buscar_personaje'

In [4]:
buscar_personaje.name
buscar_personaje.args_schema
buscar_personaje.description

'Busca la recompensa (bounty) de un personaje de One Piece por su nombre.\n\n    Intenta primero la API en vivo de One Piece y, si falla, usa una base de\n    datos local con las recompensas canónicas de la tripulación de Sombrero\n    de Paja. Devuelve un texto con el nombre y la recompensa en berries.'

### 2.2. Tool `calculadora`

Los modelos de lenguaje son malos sumando números grandes. Por eso les damos una calculadora.

Punto importante de seguridad: **nunca uses `eval()`** sobre texto que viene de un modelo, porque podría ejecutar código malicioso. Aquí usamos el módulo `ast` para analizar la expresión y permitir **solo** operaciones aritméticas (suma, resta, multiplicación, división, potencia).


In [5]:
import ast
import operator

# Operadores aritméticos permitidos: nada de funciones ni acceso a variables.
_OPERADORES = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
}


def _evaluar_nodo(nodo: ast.AST) -> float:
    """Recorre el árbol sintáctico permitiendo solo aritmética."""
    if isinstance(nodo, ast.Constant) and isinstance(nodo.value, (int, float)):
        return nodo.value
    if isinstance(nodo, ast.BinOp) and type(nodo.op) in _OPERADORES:
        return _OPERADORES[type(nodo.op)](
            _evaluar_nodo(nodo.left), _evaluar_nodo(nodo.right)
        )
    if isinstance(nodo, ast.UnaryOp) and type(nodo.op) in _OPERADORES:
        return _OPERADORES[type(nodo.op)](_evaluar_nodo(nodo.operand))
    raise ValueError("Expresión no permitida: solo se admite aritmética simple.")


@tool
def calculadora(expresion: str) -> str:
    """Evalúa una expresión aritmética y devuelve el resultado.

    Admite sumas, restas, multiplicaciones, divisiones y potencias con
    paréntesis, por ejemplo '3000000000 + 1111000000'. No admite funciones
    ni variables. Usa esta herramienta siempre que necesites operar números.
    """
    try:
        arbol = ast.parse(expresion, mode="eval")
        resultado = _evaluar_nodo(arbol.body)
        # Enteros con separador de miles: nada de notación científica que pierda cifras.
        if float(resultado).is_integer():
            texto = f"{int(resultado):,}".replace(",", ".")
        else:
            texto = str(round(resultado, 6)).replace(".", ",")
        return f"El resultado de '{expresion}' es {texto}."
    except Exception as error:
        return f"No se pudo calcular '{expresion}': {error}"


# Probamos la calculadora
print(calculadora.invoke({"expresion": "3000000000 + 1111000000 + 1032000000"}))


El resultado de '3000000000 + 1111000000 + 1032000000' es 5.143.000.000.


### 2.3. Tool `buscar_wikipedia`

La tercera tool permite al agente consultar conocimiento general. Usa la API REST de Wikipedia (con `urllib`) para traer un resumen corto de un artículo.

Fíjate en que también gestiona errores: si no hay resultados o la consulta es ambigua, devuelve un mensaje útil en lugar de lanzar una excepción.


In [6]:
import urllib.request
import urllib.parse
import json


@tool
def buscar_wikipedia(query: str) -> str:
    """Busca un resumen de un tema en Wikipedia en español.

    Útil para obtener contexto o datos generales sobre personajes, lugares
    o conceptos. Devuelve las primeras frases del artículo más relevante.
    """
    titulo = urllib.parse.quote(query.strip().replace(" ", "_"))
    url = f"https://es.wikipedia.org/api/rest_v1/page/summary/{titulo}"
    req = urllib.request.Request(
        url, headers={"User-Agent": "keepcoding-ai-lab/1.0 (curso)"}
    )
    try:
        with urllib.request.urlopen(req, timeout=15) as r:
            data = json.load(r)
        return data.get("extract") or f"Sin resumen de Wikipedia para '{query}'."
    except Exception as error:
        return f"No se pudo consultar Wikipedia para '{query}': {error}"


# Probamos la tool de Wikipedia
print(buscar_wikipedia.invoke({"query": "Zoro"}))

Los Mugiwara no Ichimi , algunas veces referidos como Mugiwara Kaizokudan , o simplemente los Mugiwara  – en traducciones en español siendo llamados Banda de Sombrero de Paja en Hispanoamérica y Banda Pirata del Sombrero de Paja o Banda del Sombrero de Paja en España –, son una tripulación pirata ficticia de la serie de manga One Piece, escrita y dibujada por Eiichirō Oda, y personajes principales de la franquicia de medios surgida de esta. La tripulación recibe el nombre a partir del apodo de su capitán y fundador, Monkey D. Luffy, alias "Sombrero de Paja".


## 3. El modelo elige una herramienta

Antes de montar el agente completo, veamos la pieza central: un modelo de chat al que le **damos a conocer** las tools con `bind_tools`.

Creamos el modelo `ChatOpenAI` con `gpt-5-mini` y le enlazamos las tres herramientas. Cuando le hagamos una pregunta, el modelo no ejecutará la tool: solo decidirá **cuál** usar y con **qué argumentos**. Esa decisión aparece en el campo `tool_calls` de su respuesta.


In [7]:
from langchain_openai import ChatOpenAI

# El modelo de chat. gpt-5-mini es rápido y barato para clase.
modelo = ChatOpenAI(model="gpt-5-mini")

# Lista de tools que el agente podrá usar.
herramientas = [buscar_personaje, calculadora, buscar_wikipedia]

# bind_tools le enseña al modelo qué herramientas existen y cómo llamarlas.
modelo_con_tools = modelo.bind_tools(herramientas)

# Pregunta simple: esperamos que el modelo elija 'buscar_personaje'.
respuesta = modelo_con_tools.invoke("¿Cuál es la recompensa de Nami?")

print(respuesta)
print("=======")
print("Texto de la respuesta:", respuesta.content or "(vacío: el modelo decidió usar una tool)")
print("=======")
print("Tool calls solicitadas:")
for llamada in respuesta.tool_calls:
    print(f"  - tool: {llamada['name']}  |  args: {llamada['args']}")


respuesta = modelo_con_tools.invoke("¿Quién es Eric Risco?")

print(respuesta)
print("=======")
print("Texto de la respuesta:", respuesta.content or "(vacío: el modelo decidió usar una tool)")
print("=======")
print("Tool calls solicitadas:")
for llamada in respuesta.tool_calls:
    print(f"  - tool: {llamada['name']}  |  args: {llamada['args']}")


content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 89, 'prompt_tokens': 336, 'total_tokens': 425, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EahUhYoGeqhs8Zly0cxH6K20KWU9J', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a1544d-2fc7-7059-9dbe-9ac0291c4dd7-0' tool_calls=[{'name': 'buscar_personaje', 'args': {'nombre': 'Nami'}, 'id': 'call_Ou7jWcuA8006bhu5vVfkeQWg', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 336, 'output_tokens': 89, 'total_tokens': 425, 'input_token_details': {'audio': 0, 'cache_read':

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 334, 'total_tokens': 424, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EahUi5QKtU7ITZ7gHXDfNYrNzctDO', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a1544d-353c-7658-b45a-5bde43abcdd3-0' tool_calls=[{'name': 'buscar_wikipedia', 'args': {'query': 'Eric Risco'}, 'id': 'call_hK8FcGh0fTpOATbDdKUNF6Tp', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 334, 'output_tokens': 90, 'total_tokens': 424, 'input_token_details': {'audio': 0, 'cache_r

## 4. El agente ReAct

`bind_tools` solo decide la llamada. Falta quien **ejecute** la tool, **devuelva** el resultado al modelo y **repita** el ciclo hasta terminar. Eso es exactamente el bucle ReAct:

```
Thought      -> el modelo razona que necesita
Action       -> llama a una tool
Observation  -> recibe el resultado de la tool
... y vuelve a empezar hasta dar la respuesta final
```

`create_agent` de `langchain.agents` (en versiones antiguas, `create_react_agent` de `langgraph.prebuilt`, deprecado en LangChain v1.0) construye este bucle por nosotros. Solo le pasamos el modelo, las tools y, opcionalmente, un `system_prompt` con instrucciones de sistema.


In [8]:
from langchain.agents import create_agent  # LangChain v1.0+ · sustituye al deprecado create_react_agent

# Instrucciones de sistema: definen el rol y el comportamiento del agente.
PROMPT_SISTEMA = (
    "Eres un experto en One Piece especializado en recompensas (bounties). "
    #"Usa 'buscar_personaje' para obtener la recompensa de cada personaje. "
    #"Usa siempre 'calculadora' para cualquier suma o operación numérica: "
    #"no calcules de cabeza. Usa 'buscar_wikipedia' para contexto general. "
    "Responde en español, de forma clara y con las cifras bien formateadas."
)

# Construimos el agente ReAct: modelo + tools + prompt de sistema.
agente = create_agent(
    model=modelo,
    tools=herramientas,
    system_prompt=PROMPT_SISTEMA,
)

# Prueba rápida con .invoke(): el agente resuelve el bucle completo internamente.
resultado = agente.invoke(
    {"messages": [{"role": "user", "content": "¿Cuál es la recompensa de Robin?"}]}
)

# El último mensaje contiene la respuesta final del agente.
print(resultado["messages"][-1].content)


La recompensa de Nico Robin es de 930.000.000 berries. (Fuente: One Piece API, recompensa actual registrada.)


## 5. Demo estrella: la recompensa total de la tripulación

Ahora la pregunta interesante:

> ¿Cuál es la recompensa total de la tripulación de Sombrero de Paja?

El agente tendrá que buscar a **cada uno de los 10 miembros**, recoger sus recompensas y **sumarlas con la calculadora**. Es un bucle ReAct largo y encadenado.

Con `.invoke()` solo veríamos la respuesta final. Para ver el razonamiento paso a paso usamos `agente.stream(...)`, que va emitiendo el estado a medida que el agente avanza por cada nodo.

Primero definimos una pequeña función de ayuda para imprimir cada paso de forma legible.


In [9]:
from langchain_core.messages import AIMessage, ToolMessage


def mostrar_paso(mensaje) -> None:
    """Imprime un mensaje del agente etiquetado como Thought, Action u Observation."""
    if isinstance(mensaje, AIMessage):
        if mensaje.tool_calls:
            # El modelo razona y decide llamar a una o varias tools.
            for llamada in mensaje.tool_calls:
                print(f"  [Action]      {llamada['name']}({llamada['args']})")
            if mensaje.content:
                print(f"  [Thought]     {mensaje.content}")
        else:
            # Mensaje sin tool calls: es la respuesta final.
            print(f"  [Respuesta]   {mensaje.content}")
    elif isinstance(mensaje, ToolMessage):
        # Resultado devuelto por una tool.
        print(f"  [Observation] {mensaje.content}")


def ejecutar_con_streaming(agente_a_usar, pregunta: str, config: dict | None = None) -> None:
    """Ejecuta el agente en modo streaming y muestra el bucle ReAct."""
    print(f"PREGUNTA: {pregunta}\n")
    entrada = {"messages": [{"role": "user", "content": pregunta}]}
    for fragmento in agente_a_usar.stream(entrada, config=config, stream_mode="values"):
        # Cada fragmento trae el historial completo: miramos el último mensaje.
        mostrar_paso(fragmento["messages"][-1])
    print("\n--- fin del bucle ---")


Ahora lanzamos la pregunta estrella. Observa en la salida como el agente encadena varias rondas de **Action -> Observation**: una búsqueda por cada miembro y, al final, una llamada a la `calculadora` para sumarlo todo.

> El total canónico post-Wano es **8.816.001.000 berries**. Comprueba a qué cifra llega el agente. Ojo: la API en vivo registra a Zoro con 1.101.000.000 (en vez de 1.111.000.000), así que con datos en vivo el total sale **8.806.001.000**. Buen recordatorio: un agente es tan fiable como sus fuentes.


In [10]:
pregunta_estrella = (
    "¿Cuál es la recompensa total de la tripulación de Sombrero de Paja después del arco de Wano?"
)

ejecutar_con_streaming(agente, pregunta_estrella)


PREGUNTA: ¿Cuál es la recompensa total de la tripulación de Sombrero de Paja después del arco de Wano?



  [Action]      buscar_personaje({'nombre': 'Monkey D. Luffy'})
  [Action]      buscar_personaje({'nombre': 'Roronoa Zoro'})
  [Action]      buscar_personaje({'nombre': 'Nami'})
  [Action]      buscar_personaje({'nombre': 'Usopp'})
  [Action]      buscar_personaje({'nombre': 'Sanji'})
  [Action]      buscar_personaje({'nombre': 'Tony Tony Chopper'})
  [Action]      buscar_personaje({'nombre': 'Nico Robin'})
  [Action]      buscar_personaje({'nombre': 'Franky'})
  [Action]      buscar_personaje({'nombre': 'Brook'})
  [Action]      buscar_personaje({'nombre': 'Jinbe'})
  [Action]      buscar_personaje({'nombre': 'Yamato'})
  [Observation] Yamato: recompensa desconocida berries (One Piece API).


  [Action]      calculadora({'expresion': '3000000000 + 1101000000 + 366000000 + 500000000 + 1032000000 + 1000 + 930000000 + 394000000 + 383000000 + 1100000000'})
  [Observation] El resultado de '3000000000 + 1101000000 + 366000000 + 500000000 + 1032000000 + 1000 + 930000000 + 394000000 + 383000000 + 1100000000' es 8.806.001.000.


  [Respuesta]   Tras el arco de Wano (según las recompensas canónicas conocidas), la suma de las recompensas de la tripulación de Sombrero de Paja es:

Recompensas individuales:
- Monkey D. Luffy: 3.000.000.000 berries  
- Roronoa Zoro: 1.101.000.000 berries  
- Nami: 366.000.000 berries  
- Usopp: 500.000.000 berries  
- Sanji: 1.032.000.000 berries  
- Tony Tony Chopper: 1.000 berries  
- Nico Robin: 930.000.000 berries  
- Franky: 394.000.000 berries  
- Brook: 383.000.000 berries  
- Jinbe: 1.100.000.000 berries  
- Yamato: sin recompensa conocida (no revelada)

Total (excluyendo a Yamato, por no tener recompensa conocida): 8.806.001.000 berries.

Si quieres, puedo actualizar el total si aparece la recompensa de Yamato o mostrar esto en formato distinto.

--- fin del bucle ---


## 6. Memoria conversacional

El agente que tenemos olvida todo entre llamadas: cada `invoke` o `stream` empieza de cero. Para que recuerde la conversación le añadimos un **checkpointer**.

`MemorySaver` guarda el estado en memoria y lo organiza por `thread_id`. Cada hilo es una conversación independiente: si reutilizas el mismo `thread_id`, el agente recuerda los mensajes anteriores.

Recompilamos el agente pasando `checkpointer=MemorySaver()`.


In [11]:
from langgraph.checkpoint.memory import MemorySaver

# Mismo agente, pero ahora con memoria.
memoria = MemorySaver()
agente_con_memoria = create_agent(
    model=modelo,
    tools=herramientas,
    system_prompt=PROMPT_SISTEMA,
    checkpointer=memoria,
)

# El thread_id identifica esta conversación concreta.
config_hilo = {"configurable": {"thread_id": "tripulación-001"}}

# Primer turno: el agente busca una recompensa concreta.
ejecutar_con_streaming(
    agente_con_memoria,
    "¿Cuál es la recompensa de Luffy?",
    config=config_hilo,
)


PREGUNTA: ¿Cuál es la recompensa de Luffy?



  [Action]      buscar_personaje({'nombre': 'Monkey D. Luffy'})
  [Observation] Monkey D Luffy: recompensa 3.000.000.000 berries (One Piece API).


  [Respuesta]   La recompensa de Monkey D. Luffy es 3.000.000.000 berries.

Fuente: One Piece API (valor canónico según la última actualización). ¿Quieres también el historial de sus recompensas anteriores o la fecha/arc en que se publicó esta cifra?

--- fin del bucle ---


### Pregunta de seguimiento

Ahora hacemos una pregunta que **solo tiene sentido con el contexto previo**: "¿Y cuánto es el doble?". No mencionamos a Luffy ni ninguna cifra.

Como usamos el **mismo `thread_id`**, el agente recuerda que hablábamos de la recompensa de Luffy, recupera ese número y lo usa con la calculadora. Si cambiaras el `thread_id`, el agente no sabría de qué le hablas.


In [12]:
# Segundo turno en el MISMO hilo: la pregunta depende del contexto anterior.
ejecutar_con_streaming(
    agente_con_memoria,
    "¿Y cuánto sería el doble de esa recompensa?",
    config=config_hilo,
)


PREGUNTA: ¿Y cuánto sería el doble de esa recompensa?



  [Action]      calculadora({'expresion': '3000000000 * 2'})
  [Observation] El resultado de '3000000000 * 2' es 6.000.000.000.


  [Respuesta]   El doble de 3.000.000.000 berries es 6.000.000.000 berries.

--- fin del bucle ---


## 7. Recap y retos

### Qué has construido

- **Tres herramientas** con `@tool`: una con *fallback* (`buscar_personaje`), una segura con `ast` (`calculadora`) y una de conocimiento (`buscar_wikipedia`).
- Has visto a un modelo **elegir tools** con `bind_tools`.
- Has montado un **agente ReAct** con `create_agent` (LangChain v1) que encadena el bucle *Thought -> Action -> Observation* hasta resolver la tarea.
- Le has dado **memoria** con `MemorySaver` y un `thread_id`.

### Ideas clave para recordar

- El **docstring** de una tool es su interfaz con el modelo: cuídalo.
- Una tool robusta **nunca lanza excepciones** hacia el agente: devuelve un mensaje de error útil.
- `stream(...)` te deja ver el razonamiento; `invoke(...)` solo da el resultado.
- Sin checkpointer no hay memoria: el `thread_id` separa conversaciones.

### Retos para practicar

1. **Nueva tool**: añade una tool `comparar_recompensas(nombre_a, nombre_b)` que indique quién tiene la recompensa más alta y por cuánta diferencia. Pruébala preguntando al agente quién es más buscado, Zoro o Sanji.
2. **Prompt de sistema**: modifica `PROMPT_SISTEMA` para que el agente responda siempre con un toque pirata y formatee las cifras con separador de miles. Observa cómo cambia su comportamiento.
3. **Memoria a prueba**: lanza la pregunta de seguimiento con un `thread_id` distinto y comprueba que el agente ya no recuerda el contexto. Luego usa `agente_con_memoria.get_state(config)` para inspeccionar el estado guardado de un hilo.
